# DengueCast v4 — Treino e Avaliação

**Dataset:** [`pedrolucassantanaf/dengue-tcc2-data`](https://huggingface.co/datasets/pedrolucassantanaf/dengue-tcc2-data)  
**Dashboard:** [modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app](https://modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app)  
**Pipeline:** v4 — 137 mesorregiões, 2001–2026, XGBoost

---

Este notebook treina XGBoost de regressão e classificação para previsão de dengue, com 3 braços de ablação (SINAN-only, INMET bruto, INMET enriquecido) e baselines obrigatórios.

In [ ]:
%%capture
!pip install huggingface_hub xgboost scikit-learn

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from huggingface_hub import hf_hub_download

IS_KAGGLE = "KAGGLE_DATA_DIR" in os.environ
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

REPO = os.environ.get("HF_REPO_ID", "pedrolucassantanaf/dengue-tcc2-data")

def hf_download(path):
    return hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)

print(f"Ambiente: {'Kaggle' if IS_KAGGLE else 'Local'}")

## 1. Carregar dados — Variante B (principal)

Treino: 2001–2022 (157.413 linhas) | Validação: 2023 | Teste: 2024–2026

In [ ]:
train = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/train.parquet"))
val = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/val.parquet"))
test = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/test.parquet"))

schema = pd.read_csv(hf_download("data/model_ready_v4/feature_schema_v4.csv"))

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Features totais: {len(schema)}")

## 2. Definir braços de ablação

Usamos o `feature_schema_v4.csv` para separar features por origem (nunca pelo nome da coluna).

In [ ]:
feats_sinan = schema[schema["category"] == "sinan"]["feature"].tolist()
feats_inmet_bruto = feats_sinan + schema[schema["category"] == "inmet_bruto"]["feature"].tolist()
feats_enriquecido = schema["feature"].tolist()

ablation_arms = {
    "sinan_only": feats_sinan,
    "inmet_bruto": feats_inmet_bruto,
    "inmet_enriquecido": feats_enriquecido,
}

for name, feats in ablation_arms.items():
    print(f"  {name}: {len(feats)} features")

TARGET = "notificacoes_t4"
TARGET_CLS = "risco_surto_t4"

## 3. Baselines obrigatórios

Antes de treinar XGBoost, calculamos baselines para referência:
- **Persistência**: notificações atuais como previsão t+4
- **Sazonal**: mesmo período do ano anterior

In [ ]:
def eval_metrics(y_true, y_pred, scale="log"):
    if scale == "log":
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
    else:
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
    return {"R²": r2, "RMSE": rmse, "MAE": mae}

y_test_log = np.log1p(test[TARGET])
y_test_orig = test[TARGET]

# Baseline 1: Persistência (notificacoes atuais como previsão de t+4)
baseline_persist_log = np.log1p(test["notificacoes"])
baseline_persist_orig = test["notificacoes"]

# Baseline 2: Sazonal (mesmo período do ano anterior)
train_val = pd.concat([train, val], ignore_index=True)
seasonal_lookup = train_val.groupby(["cod_mesorregiao", "semana_epidemiologica"])[TARGET].mean()
baseline_seasonal_orig = test.apply(
    lambda r: seasonal_lookup.get((r["cod_mesorregiao"], r["semana_epidemiologica"]), 0), axis=1
)
baseline_seasonal_log = np.log1p(baseline_seasonal_orig)

print("=== Baselines no teste ===")
print("\nPersistência (escala log):")
m = eval_metrics(y_test_log, baseline_persist_log)
print(f"  R²_log={m['R²']:.4f}, RMSE={m['RMSE']:.4f}, MAE={m['MAE']:.4f}")
m_orig = eval_metrics(y_test_orig, baseline_persist_orig)
print(f"  R²_orig={m_orig['R²']:.4f}, RMSE={m_orig['RMSE']:.0f}, MAE={m_orig['MAE']:.0f}")

print("\nSazonal (média histórica por mesorregião-semana):")
m = eval_metrics(y_test_log, baseline_seasonal_log)
print(f"  R²_log={m['R²']:.4f}, RMSE={m['RMSE']:.4f}, MAE={m['MAE']:.4f}")
m_orig = eval_metrics(y_test_orig, baseline_seasonal_orig)
print(f"  R²_orig={m_orig['R²']:.4f}, RMSE={m_orig['RMSE']:.0f}, MAE={m_orig['MAE']:.0f}")

## 4. Treinar XGBoost — Regressão (Variante B, 3 braços)

Treinamos em `log1p(notificacoes_t4)` com early stopping na validação.

In [ ]:
y_train_log = np.log1p(train[TARGET])
y_val_log = np.log1p(val[TARGET])

results = {}

for arm_name, arm_feats in ablation_arms.items():
    print(f"\n--- Treinando: B_{arm_name} ({len(arm_feats)} features) ---")
    t0 = time.time()

    model = xgb.XGBRegressor(
        n_estimators=2000,
        tree_method="hist",
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=50,
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(
        train[arm_feats], y_train_log,
        eval_set=[(val[arm_feats], y_val_log)],
        verbose=False,
    )

    elapsed = time.time() - t0
    y_pred_log = model.predict(test[arm_feats])
    y_pred_orig = np.expm1(y_pred_log)

    m_log = eval_metrics(y_test_log, y_pred_log)
    m_orig = eval_metrics(y_test_orig, y_pred_orig)

    results[f"B_{arm_name}"] = {
        "R2_log": m_log["R²"],
        "R2_orig": m_orig["R²"],
        "RMSE": m_orig["RMSE"],
        "MAE": m_orig["MAE"],
        "best_iter": model.best_iteration,
        "time_s": elapsed,
        "model": model,
    }

    print(f"  Best iter: {model.best_iteration}, Tempo: {elapsed:.0f}s")
    print(f"  R²_log={m_log['R²']:.4f}, R²_orig={m_orig['R²']:.4f}, RMSE={m_orig['RMSE']:.0f}, MAE={m_orig['MAE']:.0f}")

## 5. Tabela comparativa: Baselines vs XGBoost

In [ ]:
rows = []

# Baselines
m_p = eval_metrics(y_test_log, baseline_persist_log)
m_p_o = eval_metrics(y_test_orig, baseline_persist_orig)
rows.append({"Modelo": "Persistência (baseline)", "R²_log": m_p["R²"], "R²_orig": m_p_o["R²"], "RMSE": m_p_o["RMSE"], "MAE": m_p_o["MAE"]})

m_s = eval_metrics(y_test_log, baseline_seasonal_log)
m_s_o = eval_metrics(y_test_orig, baseline_seasonal_orig)
rows.append({"Modelo": "Sazonal (baseline)", "R²_log": m_s["R²"], "R²_orig": m_s_o["R²"], "RMSE": m_s_o["RMSE"], "MAE": m_s_o["MAE"]})

# XGBoost
for name, r in results.items():
    rows.append({"Modelo": f"XGBoost {name}", "R²_log": r["R2_log"], "R²_orig": r["R2_orig"], "RMSE": r["RMSE"], "MAE": r["MAE"]})

comparison = pd.DataFrame(rows)
comparison["R²_log"] = comparison["R²_log"].map("{:.4f}".format)
comparison["R²_orig"] = comparison["R²_orig"].map("{:.4f}".format)
comparison["RMSE"] = comparison["RMSE"].map("{:.0f}".format)
comparison["MAE"] = comparison["MAE"].map("{:.0f}".format)

print("\n=== Comparação: Baselines vs XGBoost (teste 2024–2026) ===")
print(comparison.to_string(index=False))

## 6. Classificação de risco — Variante B inmet_enriquecido

4 classes: Baixo (0), Moderado (1), Alto (2), Surto (3)

In [ ]:
y_train_cls = train[TARGET_CLS]
y_val_cls = val[TARGET_CLS]
y_test_cls = test[TARGET_CLS]

print(f"Distribuição treino: {dict(y_train_cls.value_counts().sort_index())}")
print(f"Distribuição teste:  {dict(y_test_cls.value_counts().sort_index())}")

cls_model = xgb.XGBClassifier(
    n_estimators=2000,
    tree_method="hist",
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    eval_metric="mlogloss",
    num_class=4,
    objective="multi:softprob",
    random_state=42,
    n_jobs=-1,
)

t0 = time.time()
cls_model.fit(
    train[feats_enriquecido], y_train_cls,
    eval_set=[(val[feats_enriquecido], y_val_cls)],
    verbose=False,
)
elapsed = time.time() - t0

y_pred_cls = cls_model.predict(test[feats_enriquecido])

f1_macro = f1_score(y_test_cls, y_pred_cls, average="macro")
print(f"\nF1 macro: {f1_macro:.4f}")
print(f"Tempo: {elapsed:.0f}s, Best iter: {cls_model.best_iteration}")
print("\nRelatório por classe:")
print(classification_report(y_test_cls, y_pred_cls,
                            target_names=["Baixo", "Moderado", "Alto", "Surto"]))

## 7. Verificação com valores de referência

Comparamos com os resultados do pipeline local (`46_regression_v4_report.json`).

In [ ]:
ref = json.loads(open(hf_download("data/model_ready_v4/46_regression_v4_report.json")).read())

print("=== Verificação: notebook vs referência ===")
print(f"{'Modelo':<25s} {'Notebook':>10s} {'Referência':>12s} {'Delta':>8s}")
print("-" * 58)
for arm_name in ["sinan_only", "inmet_bruto", "inmet_enriquecido"]:
    key = f"B_{arm_name}"
    nb_r2 = results[key]["R2_log"]
    ref_r2 = ref["modelos"][key]["R2_log"]
    delta = nb_r2 - ref_r2
    status = "OK" if abs(delta) < 0.005 else "DIVERGE"
    print(f"  {key:<23s} {nb_r2:>10.4f} {ref_r2:>12.4f} {delta:>+8.4f}  [{status}]")

## 8. Salvar modelos (para uso no notebook 3)

In [ ]:
os.makedirs("models_trained", exist_ok=True)

for arm_name in ablation_arms:
    key = f"B_{arm_name}"
    path = f"models_trained/model_{key}.ubj"
    results[key]["model"].save_model(path)
    print(f"Salvo: {path}")

cls_path = "models_trained/model_B_inmet_enriquecido_cls.ubj"
cls_model.save_model(cls_path)
print(f"Salvo: {cls_path}")

## 9. (Opcional) Treinar variantes A e C

Descomente a célula abaixo para treinar todas as variantes. Tempo estimado: ~20 min.

In [ ]:
# DESCOMENTE PARA EXECUTAR (leva ~20 min)

# for variant in ["a", "c"]:
#     print(f"\n{'='*60}")
#     print(f"VARIANTE {variant.upper()}")
#     print(f"{'='*60}")
#     tr = pd.read_parquet(hf_download(f"data/model_ready_v4/splits_{variant}/train.parquet"))
#     vl = pd.read_parquet(hf_download(f"data/model_ready_v4/splits_{variant}/val.parquet"))
#     te = pd.read_parquet(hf_download(f"data/model_ready_v4/splits_{variant}/test.parquet"))
#     y_tr = np.log1p(tr[TARGET])
#     y_vl = np.log1p(vl[TARGET])
#     y_te = np.log1p(te[TARGET])
#     for arm_name, arm_feats in ablation_arms.items():
#         print(f"\n--- {variant.upper()}_{arm_name} ---")
#         m = xgb.XGBRegressor(n_estimators=2000, tree_method="hist", learning_rate=0.05,
#                              max_depth=6, subsample=0.8, colsample_bytree=0.8,
#                              early_stopping_rounds=50, eval_metric="rmse", random_state=42, n_jobs=-1)
#         m.fit(tr[arm_feats], y_tr, eval_set=[(vl[arm_feats], y_vl)], verbose=False)
#         yp = m.predict(te[arm_feats])
#         r2 = r2_score(y_te, yp)
#         print(f"  R²_log={r2:.4f}, best_iter={m.best_iteration}")

## Resumo

- XGBoost supera ambos os baselines (persistência e sazonal) significativamente
- **Melhor modelo:** B_inmet_enriquecido com R²_log ≈ 0.8878
- Features INMET enriquecidas (anomalias, médias móveis, bioclimáticas) melhoram marginalmente sobre INMET bruto
- Classificação de risco funcional com 4 classes

**Próximo notebook:** [v4_03_resultados](https://www.kaggle.com/pedrolucassantanaf/denguecast-v4-03-resultados) — visualizações e análise detalhada